# Agents — Try it in PyTorch

An **optional** hands-on companion to [Chapter 12](https://learnai.robennals.org/agents). The chapter says an agent is a model, a program around it, and the loop between them, and that the model never runs anything itself. Here you write that program. It is about thirty lines.

New to PyTorch? The [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) is a quick introduction.

## About the model used here

**Qwen3-1.7B** is small enough to run free in a few minutes, and hundreds of times smaller than the models behind ChatGPT or Claude. Tool use is a trained skill with a trained format, and this model has it, which is why a model this size can drive the loop at all.

In [ ]:
!pip install -q transformers accelerate

import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
print("running on", DEVICE)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", dtype=DTYPE).to(DEVICE)
print("model loaded")

## The tools

A tool is an ordinary Python function. The docstring is not decoration: it is what the model gets told about the tool, so it is how the model knows the tool exists and what it takes.

In [ ]:
FORECAST = {
    "bristol": "cloudy until 17:00, then rain until late",
    "leeds": "dry all day, light wind",
}

def get_weather(location: str):
    """Get today's weather forecast for a place.

    Args:
        location: The town or city to look up, for example "Bristol".
    """
    return FORECAST.get(location.lower().strip(), "no forecast for that place")

def calculate(expression: str):
    """Work out the value of an arithmetic expression.

    Args:
        expression: A sum in Python syntax, for example "3 * 1249.99".
    """
    return str(eval(expression, {"__builtins__": {}}))

TOOLS = {"get_weather": get_weather, "calculate": calculate}

## What the model is actually told

The chapter says the tool list goes into the context as text. Here is that text. The chat template turns our Python functions into a description and puts it at the top of the prompt, before anything the user said.

In [ ]:
prompt_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Is it going to rain in Bristol?"}],
    tools=list(TOOLS.values()), tokenize=False, add_generation_prompt=True,
    enable_thinking=False)

print(prompt_text[:1200])

## One tool call

Ask a question the model cannot answer from memory, and watch what it writes. It is not a reply to you.

In [ ]:
def generate(messages, max_new_tokens=700, thinking=True):
    """Run the model on a conversation and return the text it writes."""
    text = tokenizer.apply_chat_template(
        messages, tools=list(TOOLS.values()), tokenize=False,
        add_generation_prompt=True, enable_thinking=thinking)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

reply = generate([{"role": "user", "content": "Is it going to rain in Bristol this afternoon?"}])
print(reply[-400:])

That `<tool_call>` block is text the model emitted, in a format post-training taught it. Nothing has run yet.

## The harness

This is the whole program. It spots a tool call, runs the function, puts the result back into the conversation as a new message, and lets the model carry on. It stops when the model writes a reply instead of another call.

In [ ]:
TOOL_CALL = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def run_agent(question, max_steps=4, show=True):
    """The loop: generate, run any tool call, feed the result back, repeat."""
    messages = [{"role": "user", "content": question}]
    if show:
        print(f"HUMAN -> MODEL: {question}")

    for step in range(max_steps):
        reply = generate(messages)
        found = TOOL_CALL.search(reply)

        if not found:
            answer = reply.split("</think>")[-1].strip()
            if show:
                print(f"MODEL -> HUMAN: {answer}")
            return answer

        call = json.loads(found.group(1))
        name, arguments = call["name"], call.get("arguments", {})
        if show:
            print(f"  MODEL -> {name} (hidden): {arguments}")

        result = TOOLS[name](**arguments)
        if show:
            print(f"  {name} -> MODEL (hidden): {result}")

        messages.append({"role": "assistant",
                         "tool_calls": [{"type": "function",
                                         "function": {"name": name,
                                                      "arguments": arguments}}]})
        messages.append({"role": "tool", "name": name, "content": result})

    return "gave up"

In [ ]:
run_agent("Is it going to rain in Bristol this afternoon?")
print()
run_agent("I have three payments of 1249.99 and one of 874. What do they come to?")

The model never ran anything. It wrote a request, our `while` loop ran the function, and the answer came back to the model as text it then read. That is the whole of what people mean by an agent.

## Skills

A skill is a tool whose result is a page of instructions. Nothing new is needed: add a function that returns a document.

In [ ]:
SKILLS = {
    "report-fault": """Reporting a fault to the council

Use this when someone wants something broken reported. Say which service it
belongs to: lighting, roads, waste or parks. Give the street name on its own.
Mark it urgent only if somebody could be hurt before it is fixed. One dark
streetlight is not urgent. A whole dark street is.""",
}

def load_skill(name: str):
    """Load the instructions for a task.

    Args:
        name: The name of the skill, one of: report-fault.
    """
    return SKILLS.get(name, "no such skill")

TOOLS["load_skill"] = load_skill

run_agent("There is a streetlight out on Fore Street. What do I need to tell the council?")

The model asked for the document because a one-line description said it existed, then answered from what came back. Nobody retrained anything.

## Memory

Memory is the same trick again: a tool that writes a note, and a tool that looks one up. What makes it memory is that the store outlives the conversation.

In [ ]:
NOTES = []

def save_memory(note: str):
    """Save something worth remembering in later conversations.

    Args:
        note: A short, factual note.
    """
    NOTES.append(note)
    return "saved"

def search_memory(query: str):
    """Look for notes saved in earlier conversations.

    Args:
        query: What to look for.
    """
    words = [w for w in query.lower().split() if len(w) > 3]
    hits = [note for note in NOTES
            if any(word[:5] in note.lower() for word in words)]
    return "\n".join(hits) if hits else "nothing saved matching that"

TOOLS["save_memory"] = save_memory
TOOLS["search_memory"] = search_memory

run_agent("Remember that I never want flights before 09:00. Save that as a note.")
print("\nthe store now holds:", NOTES)

Now throw the conversation away. The next call starts from nothing: no history, no memory of what was said, only the same tools.

In [ ]:
run_agent("Check your notes: is there anything I have told you about flight times?")

Everything that crossed the gap was one line of text, written by one tool call and fetched back by another.

## What you saw

- The tool list is text at the top of the prompt, put there by the chat template.
- A tool call is text the model writes. Our loop is what makes anything happen.
- A skill is a tool whose result is instructions.
- Memory is a tool that writes notes and a tool that reads them.
- The model is the same in all four cases. What changed was the program around it.